In [3]:
# Initial data load from csv files 

import pandas as pd

# Load log files
bid_df = pd.read_csv("bid_log_v2.csv")
impression_df = pd.read_csv("impression_log_v2.csv")
no_bid_df = pd.read_csv("no_bid_log_v2.csv")
punt_df = pd.read_csv("punt_log_v2.csv")

# Verify output
print("Bid log rows: ",len(bid_df))
print("Impression log rows:", len(impression_df))
print("No-bid log rows:", len(no_bid_df))
print("Punt log rows:", len(punt_df))

Bid log rows:  50000
Impression log rows: 33960
No-bid log rows: 15000
Punt log rows: 8000


In [8]:
# Initial data load from S3

import pandas as pd
import boto3
import os
from dotenv import load_dotenv

# Load credentials
load_dotenv()

# S3 paths for all 4 log files
BUCKET = "githubdk"
S3_PATHS = {
    "bid":        "bid/year=2026/month=04/day=16/bid_log_v2.csv",
    "impression": "impression/year=2026/month=04/day=16/impression_log_v2.csv",
    "no_bid":     "nobid/year=2026/month=04/day=16/no_bid_log_v2.csv",
    "punt":       "punt/year=2026/month=04/day=16/punt_log_v2.csv"
}

# Create S3 client
s3 = boto3.client(
    's3',
    region_name='us-east-1',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
)

# Helper function to read CSV from S3
def read_s3_csv(bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(obj['Body'])

# Load all 4 logs from S3
print("Loading data from S3...")
bid_df        = read_s3_csv(BUCKET, S3_PATHS["bid"])
impression_df = read_s3_csv(BUCKET, S3_PATHS["impression"])
no_bid_df     = read_s3_csv(BUCKET, S3_PATHS["no_bid"])
punt_df       = read_s3_csv(BUCKET, S3_PATHS["punt"])

print("✅ Bid log rows:        ", len(bid_df))
print("✅ Impression log rows: ", len(impression_df))
print("✅ No-bid log rows:     ", len(no_bid_df))
print("✅ Punt log rows:       ", len(punt_df))


Loading data from S3...
✅ Bid log rows:         50000
✅ Impression log rows:  33960
✅ No-bid log rows:      15000
✅ Punt log rows:        8000


In [4]:
# Look at the shape and first few rows of each log
print("=== BID LOG ===")
print(bid_df.shape)
bid_df.head(3)


=== BID LOG ===
(50000, 40)


,auction_id,impression_id,timestamp,publisher_id,publisher_domain,publisher_name,publisher_tier,supply_source,supply_source_type,ssp,...,ssp_fee,ssp_rev_share_pct,ecpm,auction_won,viewable,video_completed,click,conversion,loss_reason,time_to_serve_ms
0,d7554578-c6e4-43f1-a642-ad499ca87fbe,8dd8674c-86e7-4d88-8965-5b76614627d6,2024-09-17 18:49:37,pub_001,nytimes.com,The New York Times,Premium,Sports Fan Audience Pack,3P,PubMatic,...,17.81,19.0,93620.0,1,1,1,0,0,NaN,732.0
1,68816d78-49a2-42ea-b810-812bdf48701b,49ae1dbe-25b7-47f8-a82f-86736ec4a2bd,2024-09-21 23:08:36,pub_008,patch.com,Patch Local News,Long-Tail,ESPN Direct Video,1P,TripleLift,...,9.31,27.2,34260.0,1,1,0,0,0,NaN,287.0
2,39a3c724-ed97-4a26-8ac7-40757f89c692,34081225-3929-4a7d-b5e5-01053f7f2ee0,2024-09-24 18:14:43,pub_007,buzzfeed.com,BuzzFeed,Long-Tail,Programmatic Direct Pool,1P,Index Exchange,...,11.45,23.4,49020.0,1,1,1,0,0,NaN,579.0


In [7]:
# Count unique auction_ids in each log
bid_auctions = bid_df['auction_id'].nunique()
impression_auctions = impression_df['auction_id'].nunique()
no_bid_auctions = no_bid_df['auction_id'].nunique()
punt_auctions = punt_df['auction_id'].nunique()

print("Unique auctions in bid log:        ", bid_auctions)
print("Unique auctions in impression log: ", impression_auctions)
print("Unique auctions in no-bid log:     ", no_bid_auctions)
print("Unique auctions in punt log:       ", punt_auctions)

# Also check if bid log has multiple rows per auction (multiple DSPs bidding)
print("\nTotal rows in bid log:             ", len(bid_df))
print("Rows per auction (avg):            ", round(len(bid_df) / bid_auctions, 2))

Unique auctions in bid log:         50000
Unique auctions in impression log:  33960
Unique auctions in no-bid log:      15000
Unique auctions in punt log:        8000

Total rows in bid log:              50000
Rows per auction (avg):             1.0


In [13]:
# Build the supply chain funnel
total_auctions = bid_auctions + no_bid_auctions + punt_auctions

funnel = {
   'Stage': [
    '1. Total Ad Requests',
    '2. Punted by SSP (never reached DSPs)',
    '3. Bid Requests sent to DSPs',
    '4. No-Bid (DSP filtered out)',
    '5. Bids returned to Exchange',
    '6. Impression Served (Auction Won)'
]
,
    'Count': [
        total_auctions,
        punt_auctions,
        bid_auctions + no_bid_auctions,
        no_bid_auctions,
        bid_auctions,
        impression_auctions
    ]
}

funnel_df = pd.DataFrame(funnel)
funnel_df['% of Total'] = (funnel_df['Count'] / total_auctions * 100).round(1)
funnel_df['Drop from Previous'] = funnel_df['Count'].diff().fillna(0).astype(int)

print(funnel_df.to_string(index=False))


                                Stage  Count  % of Total  Drop from Previous
                 1. Total Ad Requests  73000       100.0                   0
2. Punted by SSP (never reached DSPs)   8000        11.0              -65000
         3. Bid Requests sent to DSPs  65000        89.0               57000
         4. No-Bid (DSP filtered out)  15000        20.5              -50000
         5. Bids returned to Exchange  50000        68.5               35000
   6. Impression Served (Auction Won)  33960        46.5              -16040


In [17]:
# Why is the SSP punting?
print("=== PUNT REASONS ===")
punt_reasons = punt_df['punt_reason'].value_counts()
punt_reasons_pct = (punt_reasons / len(punt_df) * 100).round(1)

punt_analysis = pd.DataFrame({
    'Count': punt_reasons,
    '% of Punts': punt_reasons_pct
})
print(punt_analysis.to_string())


=== PUNT REASONS ===
                   Count  % of Punts
punt_reason                         
TIMEOUT             2854        35.7
AD_POD_FULL         1954        24.4
PACING_THROTTLE     1638        20.5
DUPLICATE_REQUEST    954        11.9
INVALID_REQUEST      600         7.5


In [18]:
# Why are DSPs not bidding?
print("=== NO-BID REASONS ===")
no_bid_reasons = no_bid_df['no_bid_reason'].value_counts()
no_bid_reasons_pct = (no_bid_reasons / len(no_bid_df) * 100).round(1)

no_bid_analysis = pd.DataFrame({
    'Count': no_bid_reasons,
    '% of No-Bids': no_bid_reasons_pct
})
print(no_bid_analysis.to_string())


=== NO-BID REASONS ===
                    Count  % of No-Bids
no_bid_reason                          
DSP_NO_RESPONSE      4514          30.1
FLOOR_TOO_HIGH       3726          24.8
NO_MATCHING_ADS      2941          19.6
TARGETING_MISMATCH   2288          15.3
BUDGET_EXHAUSTED     1531          10.2


In [26]:
# Which publishers have the worst funnel problems?
print("=== PUNT RATE BY PUBLISHER ===")
punt_by_pub = punt_df.groupby('publisher_name').size().reset_index(name='punt_count')
bid_by_pub = bid_df.groupby('publisher_name').size().reset_index(name='bid_count')

pub_analysis = bid_by_pub.merge(punt_by_pub, on='publisher_name', how='outer').fillna(0)
pub_analysis['punt_count'] = pub_analysis['punt_count'].astype(int)
pub_analysis['punt_rate_%'] = (pub_analysis['punt_count'] / 
                               (pub_analysis['bid_count'] + pub_analysis['punt_count']) * 100).round(1)
pub_analysis = pub_analysis.sort_values('punt_rate_%', ascending=False)
print(pub_analysis.to_string(index=False))


=== PUNT RATE BY PUBLISHER ===
     publisher_name  bid_count  punt_count  punt_rate_%
The Weather Channel       4906         837         14.6
             Forbes       5001         833         14.3
       ESPN Digital       5050         823         14.0
           BuzzFeed       5051         812         13.8
 The New York Times       4975         794         13.8
    Bleacher Report       4902         779         13.7
   Patch Local News       5064         801         13.7
             Reddit       4985         790         13.7
         TechCrunch       5055         790         13.5
           HuffPost       5011         741         12.9


In [27]:
print("=== NO-BID RATE BY PUBLISHER ===")
no_bid_by_pub = no_bid_df.groupby('publisher_name').size().reset_index(name='no_bid_count')

pub_full = pub_analysis.merge(no_bid_by_pub, on='publisher_name', how='outer').fillna(0)
pub_full['no_bid_count'] = pub_full['no_bid_count'].astype(int)
pub_full['no_bid_rate_%'] = (pub_full['no_bid_count'] / 
                             (pub_full['bid_count'] + pub_full['no_bid_count']) * 100).round(1)
pub_full = pub_full.sort_values('no_bid_rate_%', ascending=False)
print(pub_full[['publisher_name', 'bid_count', 'no_bid_count', 'no_bid_rate_%']].to_string(index=False))


=== NO-BID RATE BY PUBLISHER ===
     publisher_name  bid_count  no_bid_count  no_bid_rate_%
    Bleacher Report       4902          1518           23.6
             Forbes       5001          1540           23.5
           HuffPost       5011          1537           23.5
           BuzzFeed       5051          1534           23.3
   Patch Local News       5064          1542           23.3
The Weather Channel       4906          1488           23.3
             Reddit       4985          1502           23.2
         TechCrunch       5055          1487           22.7
 The New York Times       4975          1416           22.2
       ESPN Digital       5050          1436           22.1


In [1]:
import boto3
import json
from dotenv import load_dotenv
import os

# Load credentials from .env file
load_dotenv()

# Create Bedrock client
bedrock = boto3.client(
    service_name='bedrock-runtime',
    region_name='us-east-1',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
)

print("Bedrock client created successfully")


Bedrock client created successfully


In [4]:
# List available Claude models in your account
import boto3

bedrock_mgmt = boto3.client(
    service_name='bedrock',
    region_name='us-east-1',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
)

response = bedrock_mgmt.list_foundation_models(byProvider='Anthropic')
for model in response['modelSummaries']:
    if model.get('modelLifecycle', {}).get('status') == 'ACTIVE':
        print(model['modelId'], '|', model.get('modelLifecycle', {}).get('status'))


anthropic.claude-haiku-4-5-20251001-v1:0 | ACTIVE
anthropic.claude-sonnet-4-6 | ACTIVE
anthropic.claude-opus-4-6-v1 | ACTIVE
anthropic.claude-opus-4-7 | ACTIVE
anthropic.claude-sonnet-4-5-20250929-v1:0 | ACTIVE
anthropic.claude-opus-4-1-20250805-v1:0 | ACTIVE
anthropic.claude-opus-4-5-20251101-v1:0 | ACTIVE


In [7]:
# Package our funnel findings as context for Claude
funnel_summary = """
You are an expert programmatic advertising analyst. 
Analyze this supply chain funnel data and provide clear, actionable insights.

SUPPLY CHAIN FUNNEL:
- Total Ad Requests: 73,000
- Punted by SSP (never reached DSPs): 8,000 (11%)
- Bid Requests sent to DSPs: 65,000 (89%)
- No-Bid from DSPs: 15,000 (20.5%)
- Bids returned to Exchange: 50,000 (68.5%)
- Impressions Served: 33,960 (46.5%)

PUNT REASONS (why SSP blocked before sending to DSPs):
- TIMEOUT: 2,854 (35.7%) — SSP processing too slow
- AD_POD_FULL: 1,954 (24.4%) — video ad slot already filled
- PACING_THROTTLE: 1,638 (20.5%) — smart throttling by SSP
- DUPLICATE_REQUEST: 954 (11.9%) — duplicate requests
- INVALID_REQUEST: 600 (7.5%) — malformed requests

NO-BID REASONS (why DSPs declined after receiving request):
- DSP_NO_RESPONSE: 4,514 (30.1%) — DSP timeout, no response in time
- FLOOR_TOO_HIGH: 3,726 (24.8%) — publisher floor price too high for DSPs
- NO_MATCHING_ADS: 2,941 (19.6%) — DSP has no campaign for this inventory format
- TARGETING_MISMATCH: 2,288 (15.3%) — audience doesn't match DSP campaign targeting
- BUDGET_EXHAUSTED: 1,531 (10.2%) — DSP daily budget spent

PUBLISHER ANALYSIS:
- Punt rates are uniform across all publishers (12.9% - 14.6%) suggesting systemic SSP issue
- No-bid rates are uniform across all publishers (22.1% - 23.6%) suggesting demand portfolio gap

Please provide:
1. Top 3 problems in priority order with business impact
2. Root cause for each problem
3. Specific recommended actions for the supply-side team
"""

# Call Claude via Bedrock with updated model ID
response = bedrock.invoke_model(
    modelId='us.anthropic.claude-haiku-4-5-20251001-v1:0',
    body=json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 2048,
        "messages": [
            {"role": "user", "content": funnel_summary}
        ]
    })
)

# Parse and print the response
result = json.loads(response['body'].read())
print(result['content'][0]['text'])


# Supply Chain Funnel Analysis & Recommendations

## KEY FINDING
Your funnel loses **39% of potential revenue** (40,000 impressions) between ad request and served impression. The uniform distribution across publishers indicates **systemic infrastructure and demand issues**, not publisher-specific problems.

---

## TOP 3 PROBLEMS (Priority Order)

### 🔴 PROBLEM #1: DSP Response Failures
**Business Impact:** 4,514 lost impressions (6.2% of total requests) + 15,000 no-bids = **26.5% total demand loss**

**Root Cause:**
- DSP timeout (30.1% of no-bids) suggests SSP broadcast window is too tight
- Combined with 20.5% floor price rejections, DSPs lack time to evaluate + respond competitively
- Possible: SSP timeout settings too aggressive or DSP API endpoints underperforming

**Recommended Actions:**
1. **Audit SSP timeout settings** — Compare against industry standard (typically 100-150ms). If below 100ms, increase incrementally and measure impact.
2. **Profile DSP latency by partner** — I

In [9]:
import time
import json

# ── 1. COMPUTE METRICS FROM DATAFRAMES (live, not hardcoded) ──────────────────

total_auctions   = len(bid_df) + len(no_bid_df) + len(punt_df)
punt_count       = len(punt_df)
no_bid_count     = len(no_bid_df)
bid_count        = len(bid_df)
impression_count = len(impression_df)

punt_rate    = round(punt_count / total_auctions * 100, 1)
no_bid_rate  = round(no_bid_count / (bid_count + no_bid_count) * 100, 1)
fill_rate    = round(impression_count / total_auctions * 100, 1)
bid_win_rate = round(impression_count / bid_count * 100, 1)

# Top punt reasons
top_punt = punt_df['punt_reason'].value_counts().head(3)
punt_reasons_text = "\n".join([
    f"  - {reason}: {count} ({round(count/punt_count*100,1)}%)"
    for reason, count in top_punt.items()
])

# Top no-bid reasons
top_nobid = no_bid_df['no_bid_reason'].value_counts().head(3)
nobid_reasons_text = "\n".join([
    f"  - {reason}: {count} ({round(count/no_bid_count*100,1)}%)"
    for reason, count in top_nobid.items()
])

# ── 2. BUILD DYNAMIC PROMPT ───────────────────────────────────────────────────

prompt = f"""
You are an expert programmatic advertising analyst specializing in supply-side analytics.
Analyze this supply chain funnel data and provide clear, actionable insights.

SUPPLY CHAIN FUNNEL (live data):
- Total Ad Requests:              {total_auctions:,}
- Punted by SSP (never sent):     {punt_count:,}  ({punt_rate}%)
- Bid Requests sent to DSPs:      {bid_count + no_bid_count:,}  ({round((bid_count+no_bid_count)/total_auctions*100,1)}%)
- No-Bid from DSPs:               {no_bid_count:,}  ({no_bid_rate}% of sent)
- Bids returned to Exchange:      {bid_count:,}  ({round(bid_count/total_auctions*100,1)}%)
- Impressions Served:             {impression_count:,}  ({fill_rate}%)
- Bid Win Rate:                   {bid_win_rate}%

TOP PUNT REASONS (SSP blocked before DSPs):
{punt_reasons_text}

TOP NO-BID REASONS (DSPs declined after receiving request):
{nobid_reasons_text}

PUBLISHER ANALYSIS:
- Punt rates uniform across publishers ({punt_rate-2}% - {punt_rate+2}%) → systemic SSP issue
- No-bid rates uniform across publishers → demand portfolio gap

Please provide:
1. Top 3 problems in priority order with business impact
2. Root cause for each problem  
3. Specific recommended actions for the supply-side team
4. A summary metrics table with current values, targets and timeline
"""

# ── 3. CALL BEDROCK WITH TIMING ───────────────────────────────────────────────

print("Sending to Claude...")
start_time = time.time()

response = bedrock.invoke_model(
    modelId='us.anthropic.claude-haiku-4-5-20251001-v1:0',
    body=json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 2048,
        "messages": [{"role": "user", "content": prompt}]
    })
)

end_time = time.time()
latency_ms = round((end_time - start_time) * 1000)

# ── 4. PARSE RESPONSE + TRACK COST ───────────────────────────────────────────

result = json.loads(response['body'].read())
answer = result['content'][0]['text']

# Token usage (from Bedrock response metadata)
input_tokens  = result['usage']['input_tokens']
output_tokens = result['usage']['output_tokens']

# Claude Haiku pricing (us-east-1, as of 2025)
# $0.00025 per 1K input tokens, $0.00125 per 1K output tokens
input_cost  = round((input_tokens  / 1000) * 0.00025, 6)
output_cost = round((output_tokens / 1000) * 0.00125, 6)
total_cost  = round(input_cost + output_cost, 6)

# ── 5. PRINT EVALUATION SUMMARY ──────────────────────────────────────────────

print("\n" + "="*60)
print("📊 EVALUATION METRICS")
print("="*60)
print(f"⏱  Latency:        {latency_ms} ms")
print(f"🔤  Input tokens:   {input_tokens:,}")
print(f"🔤  Output tokens:  {output_tokens:,}")
print(f"💰  Input cost:     ${input_cost}")
print(f"💰  Output cost:    ${output_cost}")
print(f"💰  Total cost:     ${total_cost}")
print("="*60)

# ── 6. PRINT CLAUDE'S ANSWER ─────────────────────────────────────────────────

print("\n📋 CLAUDE'S ANALYSIS:\n")
print(answer)


Sending to Claude...

📊 EVALUATION METRICS
⏱  Latency:        15060 ms
🔤  Input tokens:   424
🔤  Output tokens:  1,618
💰  Input cost:     $0.000106
💰  Output cost:    $0.002022
💰  Total cost:     $0.002128

📋 CLAUDE'S ANALYSIS:

# Supply-Side Analytics: Critical Findings & Action Plan

## 1. TOP 3 PROBLEMS (Priority Order)

### 🔴 PROBLEM #1: Low Impression Fill Rate (46.5%)
**Business Impact:** ~39,040 missed impressions (53.5% of requests) = significant revenue leakage

**Root Cause Analysis:**
- Only 68.5% of bid requests convert to bids (50K bids from 65K sent)
- Of those bids, only 67.9% win impressions (33,960 served)
- **Multiplier effect:** 0.685 × 0.679 = 46.5% → compounding loss at each funnel stage

**Specific Actions:**
1. **Audit DSP Response SLAs** – 30.1% of no-bids are timeouts
   - Implement 200ms response timeout enforcement
   - Establish performance contracts with underperforming DSPs
   - Add real-time monitoring dashboard for response latency by DSP

2. **Floor Pri

In [42]:
## Practice cell - Delete after that

import time
import json

total_auctions   = len(bid_df) + len(no_bid_df) + len(punt_df)
punt_count       = len(punt_df)
no_bid_count     = len(no_bid_df)
bid_count        = len(bid_df)
impression_count = len(impression_df)

punt_rate    = round(punt_count / total_auctions * 100, 1)
no_bid_rate  = round(no_bid_count / (bid_count + no_bid_count) * 100, 1)
fill_rate    = round(impression_count / total_auctions * 100, 1)
bid_win_rate = round(impression_count / bid_count * 100, 1)
punt_reasons = punt_df['punt_reason'].value_counts().head(3)
print(punt_reasons)
punt_reasons_text ="\n".join([f" - {reason}: {count} ({round(count/punt_count*100,1)}%)"
  for reason, count in top_punt.items()
])
print(punt_reasons_text)
top_punt = punt_df['punt_reason'].value_counts().head(3)
print(top_punt)

# ═══════════════════════════════════════════════════════════════
# SEGMENT 2: BUILD DYNAMIC PROMPT
# ═══════════════════════════════════════════════════════════════

prompt = f"""
You are an expert programmatic advertising analyst specializing in supply-side analytics.
Analyze this supply chain funnel data and provide clear, actionable insights.

SUPPLY CHAIN FUNNEL (live data):
- Total Ad Requests:              {total_auctions:,}
- Punted by SSP (never sent):     {punt_count:,} ({punt_rate}%)
- Bid Requests sent to DSPs:      {bid_count + no_bid_count:,} ({round((bid_count+no_bid_count)/total_auctions*100,1)}%)
- No-Bid from DSPs:               {no_bid_count:,} ({no_bid_rate}% of sent)
- Bids returned to Exchange:      {bid_count:,} ({round(bid_count/total_auctions*100,1)}%)
- Impressions Served:             {impression_count:,} ({fill_rate}%)
- Bid Win Rate:                   {bid_win_rate}%

TOP PUNT REASONS (SSP blocked before DSPs):
{punt_reasons_text}

TOP NO-BID REASONS (DSPs declined after receiving request):
{nobid_reasons_text}

PUBLISHER ANALYSIS:
- Punt rates uniform across all publishers ({punt_rate-2}% - {punt_rate+2}%)
- No-bid rates uniform across all publishers — demand portfolio gap

Please provide:
1. Top 3 problems in priority order with business impact
2. Root cause for each problem
3. Specific recommended actions for the supply-side team
4. A summary metrics table with current values, targets and timeline
"""

# ═══════════════════════════════════════════════════════════════
# SEGMENT 3: CALL BEDROCK WITH TIMING
# ═══════════════════════════════════════════════════════════════

print("Sending to Claude for analysis...")
start_time = time.time()

response = bedrock.invoke_model(
    modelId='us.anthropic.claude-haiku-4-5-20251001-v1:0',
    body=json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 2048,
        "messages": [{"role": "user", "content": prompt}]
    })
)

end_time = time.time()
latency_ms = round((end_time - start_time) * 1000)

result       = json.loads(response['body'].read())
answer       = result['content'][0]['text']
input_tokens  = result['usage']['input_tokens']
output_tokens = result['usage']['output_tokens']

print(answer)
#print(answer)


punt_reason
TIMEOUT            2854
AD_POD_FULL        1954
PACING_THROTTLE    1638
Name: count, dtype: int64
 - TIMEOUT: 2854 (35.7%)
 - AD_POD_FULL: 1954 (24.4%)
 - PACING_THROTTLE: 1638 (20.5%)
punt_reason
TIMEOUT            2854
AD_POD_FULL        1954
PACING_THROTTLE    1638
Name: count, dtype: int64
Sending to Claude for analysis...


IndexError: list index out of range

In [27]:
import time
import json

# ═══════════════════════════════════════════════════════════════
# SEGMENT 1: COMPUTE METRICS FROM DATAFRAMES (live, not hardcoded)
# ═══════════════════════════════════════════════════════════════

total_auctions   = len(bid_df) + len(no_bid_df) + len(punt_df)
punt_count       = len(punt_df)
no_bid_count     = len(no_bid_df)
bid_count        = len(bid_df)
impression_count = len(impression_df)

punt_rate    = round(punt_count / total_auctions * 100, 1)
no_bid_rate  = round(no_bid_count / (bid_count + no_bid_count) * 100, 1)
fill_rate    = round(impression_count / total_auctions * 100, 1)
bid_win_rate = round(impression_count / bid_count * 100, 1)

# Top punt reasons
top_punt = punt_df['punt_reason'].value_counts().head(3)
punt_reasons_text = "\n".join([f"  - {reason}: {count} ({round(count/punt_count*100,1)}%)"
    for reason, count in top_punt.items()
])

# Top no-bid reasons
top_nobid = no_bid_df['no_bid_reason'].value_counts().head(3)
nobid_reasons_text = "\n".join([
    f"  - {reason}: {count} ({round(count/no_bid_count*100,1)}%)"
    for reason, count in top_nobid.items()
])

# ═══════════════════════════════════════════════════════════════
# SEGMENT 2: BUILD DYNAMIC PROMPT
# ═══════════════════════════════════════════════════════════════

prompt = f"""
You are an expert programmatic advertising analyst specializing in supply-side analytics.
Analyze this supply chain funnel data and provide clear, actionable insights.

SUPPLY CHAIN FUNNEL (live data):
- Total Ad Requests:              {total_auctions:,}
- Punted by SSP (never sent):     {punt_count:,} ({punt_rate}%)
- Bid Requests sent to DSPs:      {bid_count + no_bid_count:,} ({round((bid_count+no_bid_count)/total_auctions*100,1)}%)
- No-Bid from DSPs:               {no_bid_count:,} ({no_bid_rate}% of sent)
- Bids returned to Exchange:      {bid_count:,} ({round(bid_count/total_auctions*100,1)}%)
- Impressions Served:             {impression_count:,} ({fill_rate}%)
- Bid Win Rate:                   {bid_win_rate}%

TOP PUNT REASONS (SSP blocked before DSPs):
{punt_reasons_text}

TOP NO-BID REASONS (DSPs declined after receiving request):
{nobid_reasons_text}

PUBLISHER ANALYSIS:
- Punt rates uniform across all publishers ({punt_rate-2}% - {punt_rate+2}%)
- No-bid rates uniform across all publishers — demand portfolio gap

Please provide:
1. Top 3 problems in priority order with business impact
2. Root cause for each problem
3. Specific recommended actions for the supply-side team
4. A summary metrics table with current values, targets and timeline
"""

# ═══════════════════════════════════════════════════════════════
# SEGMENT 3: CALL BEDROCK WITH TIMING
# ═══════════════════════════════════════════════════════════════

print("Sending to Claude for analysis...")
start_time = time.time()

response = bedrock.invoke_model(
    modelId='us.anthropic.claude-haiku-4-5-20251001-v1:0',
    body=json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 2048,
        "messages": [{"role": "user", "content": prompt}]
    })
)

end_time = time.time()
latency_ms = round((end_time - start_time) * 1000)

result       = json.loads(response['body'].read())
answer       = result['content'][0]['text']
input_tokens  = result['usage']['input_tokens']
output_tokens = result['usage']['output_tokens']

# Claude Haiku pricing (us-east-1, 2025)
input_cost  = round((input_tokens  / 1000) * 0.00025, 6)
output_cost = round((output_tokens / 1000) * 0.00125, 6)
total_cost  = round(input_cost + output_cost, 6)

# ═══════════════════════════════════════════════════════════════
# SEGMENT 4: LLM-AS-JUDGE EVALUATION
# Sends a second Claude call to evaluate the first response
# Scores: Relevance, Faithfulness (hallucination), Actionability
# ═══════════════════════════════════════════════════════════════

print("Sending to Claude for self-evaluation...")
eval_start = time.time()

eval_prompt = f"""
You are an AI evaluation expert. Evaluate the following AI response against the source data provided.

SOURCE DATA PROVIDED TO AI:
{prompt}

AI RESPONSE TO EVALUATE:
{answer}

Score the response on these 3 dimensions (0.0 to 1.0 each):

1. RELEVANCE (0.0-1.0)
   Did the response directly address the supply chain funnel problems?
   1.0 = fully relevant, 0.0 = completely off-topic

2. FAITHFULNESS (0.0-1.0) — hallucination detection
   Are all numbers, percentages and claims in the response 
   supported by the source data provided?
   Check every specific number mentioned.
   1.0 = all claims grounded in source data
   0.5 = some claims unsupported or slightly wrong
   0.0 = significant hallucination detected
   List any unsupported or incorrect claims found.

3. ACTIONABILITY (0.0-1.0)
   Are the recommendations specific and useful for a supply-side team?
   1.0 = very specific, immediately actionable
   0.0 = vague, generic, not useful

Respond in this exact format:
RELEVANCE: [score]
FAITHFULNESS: [score]
ACTIONABILITY: [score]
OVERALL: [average of three scores]
HALLUCINATION_ISSUES: [list any unsupported claims, or "None detected"]
SUMMARY: [one sentence overall assessment]
"""

eval_response = bedrock.invoke_model(
    modelId='us.anthropic.claude-haiku-4-5-20251001-v1:0',
    body=json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 512,
        "messages": [{"role": "user", "content": eval_prompt}]
    })
)

eval_end = time.time()
eval_latency_ms = round((eval_end - eval_start) * 1000)

eval_result       = json.loads(eval_response['body'].read())
eval_answer       = eval_result['content'][0]['text']
eval_input_tokens  = eval_result['usage']['input_tokens']
eval_output_tokens = eval_result['usage']['output_tokens']
eval_cost = round(
    (eval_input_tokens / 1000) * 0.00025 +
    (eval_output_tokens / 1000) * 0.00125, 6
)

# ═══════════════════════════════════════════════════════════════
# SEGMENT 5: PRINT FULL EVALUATION DASHBOARD
# ═══════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("📊 SYSTEM EVALUATION")
print("="*60)
print(f"⏱  Analysis latency:     {latency_ms} ms")
print(f"⏱  Evaluation latency:   {eval_latency_ms} ms")
print(f"🔤  Analysis tokens:      {input_tokens:,} in / {output_tokens:,} out")
print(f"🔤  Evaluation tokens:    {eval_input_tokens:,} in / {eval_output_tokens:,} out")
print(f"💰  Analysis cost:        ${total_cost}")
print(f"💰  Evaluation cost:      ${eval_cost}")
print(f"💰  Total cost this run:  ${round(total_cost + eval_cost, 6)}")

print("\n" + "="*60)
print("🎯 RESPONSE EVALUATION (LLM-as-Judge)")
print("="*60)
print(eval_answer)

print("\n" + "="*60)
print("📋 CLAUDE'S ANALYSIS")
print("="*60)
print(answer)


Sending to Claude for analysis...
Sending to Claude for self-evaluation...

📊 SYSTEM EVALUATION
⏱  Analysis latency:     16067 ms
⏱  Evaluation latency:   5506 ms
🔤  Analysis tokens:      419 in / 1,778 out
🔤  Evaluation tokens:    2,537 in / 435 out
💰  Analysis cost:        $0.002328
💰  Evaluation cost:      $0.001178
💰  Total cost this run:  $0.003506

🎯 RESPONSE EVALUATION (LLM-as-Judge)
RELEVANCE: 0.95

FAITHFULNESS: 0.70

ACTIONABILITY: 0.85

OVERALL: 0.83

HALLUCINATION_ISSUES:
1. **Estimated timeout threshold of "~150ms" (Section 3, Metrics Table)** — Not provided in source data. AI invented this baseline estimate.
2. **"Assume avg CPM × 15K impressions foregone"** — Placeholder language, but frames revenue loss without source data to calculate actual CPM or dollar impact.
3. **"Likely 2-3 DSPs" in root cause analysis** — Speculation presented as evidence. Source data states "uniform no-bid rates" but does NOT explicitly identify DSP count.
4. **"Industry benchmarks: 55-65% for 

In [49]:
import faiss
import numpy as np
import yaml
from sentence_transformers import SentenceTransformer

# ── 1. LOAD SEMANTIC LAYER ────────────────────────────────────────────────────
print("Loading semantic layer...")
with open("semanticlayer.yaml", "r") as f:
    semantic_layer = yaml.safe_load(f)

# ── 2. CHUNK THE SEMANTIC LAYER INTO SEARCHABLE PIECES ───────────────────────
# Each chunk = one meaningful piece of context Claude might need
chunks = []

# Add domain context
chunks.append({
    "id": "domain_overview",
    "text": semantic_layer['domain']['description']
})

# Add funnel stages
for stage in semantic_layer['domain']['funnel_stages']:
    chunks.append({
        "id": f"funnel_stage_{stage['stage'].replace(' ', '_')}",
        "text": f"Funnel stage: {stage['stage']}. {stage['description']}"
    })

# Add table descriptions
for table in semantic_layer['tables']:
    chunks.append({
        "id": f"table_{table['name']}",
        "text": f"Table: {table['name']}. {table['description']}"
    })
    # Add each column as its own chunk
    for col in table['columns']:
        chunks.append({
            "id": f"{table['name']}.{col['name']}",
            "text": f"Column {col['name']} in {table['name']}: {col['description']}"
        })

# Add metrics
for metric in semantic_layer['metrics']:
    chunks.append({
        "id": f"metric_{metric['name']}",
        "text": f"Metric: {metric['name']}. {metric['description']} Formula: {metric['formula']}"
    })

# Add punt reasons and no-bid reasons (key domain knowledge)
for table in semantic_layer['tables']:
    for col in table['columns']:
        if col['name'] in ['punt_reason', 'no_bid_reason']:
            chunks.append({
                "id": f"reasons_{col['name']}",
                "text": f"Possible values for {col['name']}: {col['description']}"
            })

print(f"Created {len(chunks)} chunks from semantic layer")

# ── 3. CONVERT CHUNKS TO EMBEDDINGS ──────────────────────────────────────────
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating embeddings...")
texts = [chunk['text'] for chunk in chunks]
embeddings = embedder.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')

print(f"Embedding shape: {embeddings.shape}")

# ── 4. BUILD FAISS INDEX ──────────────────────────────────────────────────────
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"✅ Vector DB built with {index.ntotal} vectors")
print(f"   Embedding dimensions: {dimension}")


Loading semantic layer...
Created 153 chunks from semantic layer
Loading embedding model...
Generating embeddings...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embedding shape: (153, 384)
✅ Vector DB built with 153 vectors
   Embedding dimensions: 384


In [47]:
import yaml
try:
    with open("semanticlayer.yaml", "r") as f:
        test = yaml.safe_load(f)
    print("✅ YAML is valid")
except yaml.YAMLError as e:
    print(f"❌ Error: {e}")



✅ YAML is valid


In [50]:
# ── VIEW WHAT'S STORED IN THE VECTOR DB ──────────────────────────────────────

print(f"Total vectors in FAISS index: {index.ntotal}")
print(f"Vector dimensions: {index.d}")
print()

# Show all chunks as a readable table
print("=" * 80)
print(f"{'#':<5} {'CHUNK ID':<45} {'TEXT PREVIEW':<50}")
print("=" * 80)

for i, chunk in enumerate(chunks):
    preview = chunk['text'][:60].replace('\n', ' ')
    print(f"{i:<5} {chunk['id']:<45} {preview}...")

print()
print(f"Total: {len(chunks)} chunks")


Total vectors in FAISS index: 153
Vector dimensions: 384

#     CHUNK ID                                      TEXT PREVIEW                                      
0     domain_overview                               This semantic layer describes the end-to-end programmatic ad...
1     funnel_stage_Ad_Request                       Funnel stage: Ad Request. Publisher sends an ad request to t...
2     funnel_stage_Punt                             Funnel stage: Punt. SSP/Ad Exchange evaluates the request an...
3     funnel_stage_Bid_Request                      Funnel stage: Bid Request. SSP sends the request to one or m...
4     funnel_stage_DSP_Evaluation                   Funnel stage: DSP Evaluation. DSP internally filters candida...
5     funnel_stage_No-Bid                           Funnel stage: No-Bid. DSP received the bid request but chose...
6     funnel_stage_Bid_Returned                     Funnel stage: Bid Returned. DSP submits a bid back to the Ad...
7     funnel_stage_Auction 

In [51]:
def search_vector_db(question, top_k=5):
    """
    Search the FAISS vector DB for chunks most relevant to the question.
    Returns top_k most similar chunks.
    """
    # Convert question to embedding (same model used to build the index)
    question_vector = embedder.encode([question])
    question_vector = np.array(question_vector).astype('float32')

    # Search FAISS — returns distances and indices of top_k matches
    distances, indices = index.search(question_vector, top_k)

    # Retrieve the matching chunks
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "rank":     i + 1,
            "chunk_id": chunks[idx]['id'],
            "distance": round(float(distances[0][i]), 4),
            "text":     chunks[idx]['text']
        })
    return results

# ── TEST WITH 3 DIFFERENT QUESTIONS ──────────────────────────────────────────

test_questions = [
    "Why is fill rate low?",
    "What does punt mean in programmatic advertising?",
    "How is eCPM calculated?"
]

for question in test_questions:
    print(f"\n{'='*70}")
    print(f"QUESTION: {question}")
    print(f"{'='*70}")
    results = search_vector_db(question, top_k=3)
    for r in results:
        print(f"\nRank {r['rank']} | Distance: {r['distance']} | ID: {r['chunk_id']}")
        print(f"  {r['text'][:150]}...")



QUESTION: Why is fill rate low?

Rank 1 | Distance: 1.2206 | ID: metric_fill_rate
  Metric: fill_rate. Percentage of total ad requests that resulted in a served impression. The primary supply chain health metric. Low fill rate indicat...

Rank 2 | Distance: 1.3895 | ID: bid_log.floor_price
  Column floor_price in bid_log: Minimum CPM price set by the publisher below which no bid will win. Expressed in USD per thousand impressions. High flo...

Rank 3 | Distance: 1.5407 | ID: funnel_stage_No-Bid
  Funnel stage: No-Bid. DSP received the bid request but chose not to respond with a bid. Reasons include no matching ads, targeting mismatch, floor pri...

QUESTION: What does punt mean in programmatic advertising?

Rank 1 | Distance: 0.7325 | ID: table_punt_log
  Table: punt_log. Records every ad request that the SSP/Ad Exchange decided NOT to send to any DSP. This is a pure supply-side decision made before any...

Rank 2 | Distance: 0.8717 | ID: punt_log.advertiser
  Column advertiser in pun

In [52]:
import json
import time
import datetime

def rag_answer(question, top_k=3):
    """
    Full RAG pipeline:
    1. Retrieve relevant chunks from vector DB
    2. Build prompt with retrieved context
    3. Call Claude with tight token limit
    4. Save eval result to S3
    """
    # ── RETRIEVE ─────────────────────────────────────────────
    retrieved = search_vector_db(question, top_k=top_k)
    context = "\n".join([f"- {r['text'][:200]}" for r in retrieved])

    # ── BUILD PROMPT ─────────────────────────────────────────
    prompt = f"""Use only the context below to answer the question. Be concise.

CONTEXT:
{context}

QUESTION: {question}
ANSWER:"""

    # ── CALL CLAUDE ──────────────────────────────────────────
    start = time.time()
    response = bedrock.invoke_model(
        modelId='us.anthropic.claude-haiku-4-5-20251001-v1:0',
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 300,
            "messages": [{"role": "user", "content": prompt}]
        })
    )
    latency = round((time.time() - start) * 1000)
    result  = json.loads(response['body'].read())
    answer  = result['content'][0]['text']
    tokens_in  = result['usage']['input_tokens']
    tokens_out = result['usage']['output_tokens']
    cost = round((tokens_in/1000)*0.00025 + (tokens_out/1000)*0.00125, 6)

    # ── SAVE EVAL TO S3 (Eval doesn't include LLM as a judge to save tokens) ───────────────────────────────────────
    eval_record = {
        "timestamp":   datetime.datetime.now().isoformat(),
        "question":    question,
        "retrieved_chunks": [r['chunk_id'] for r in retrieved],
        "latency_ms":  latency,
        "tokens_in":   tokens_in,
        "tokens_out":  tokens_out,
        "cost_usd":    cost,
        "answer":      answer
    }
    date_str = datetime.datetime.now().strftime("%Y-%m-%d")
    ts       = datetime.datetime.now().strftime("%H%M%S")
    s3_key   = f"eval_results/date={date_str}/{ts}.json"
    s3.put_object(
        Bucket="githubdk",
        Key=s3_key,
        Body=json.dumps(eval_record, indent=2)
    )

    # ── PRINT RESULTS ─────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    print(f"A: {answer}")
    print(f"\n📊 tokens: {tokens_in} in / {tokens_out} out | cost: ${cost} | latency: {latency}ms")
    print(f"💾 saved: s3://githubdk/{s3_key}")

# ── TEST WITH 2 QUESTIONS ─────────────────────────────────────
rag_answer("Why is fill rate low?")
rag_answer("What does punt mean in programmatic advertising?")



Q: Why is fill rate low?
A: Based on the context, low fill rate can be caused by:

1. **Demand gaps** - insufficient buyer interest
2. **Pricing misalignment** - floor prices set too high, reducing the number of winning bids
3. **Infrastructure issues**
4. **No-Bid situations** - DSPs declining to bid due to:
   - No matching ads
   - Targeting mismatch
   - Floor price too high
   - Budget exhausted
   - DSP timeout

📊 tokens: 182 in / 111 out | cost: $0.000184 | latency: 2729ms
💾 saved: s3://githubdk/eval_results/date=2026-05-03/101028.json

Q: What does punt mean in programmatic advertising?
A: Based on the context, a punt in programmatic advertising means an ad request that the SSP (Supply-Side Platform) or Ad Exchange decided **NOT to send to any DSP** (Demand-Side Platform). This is a supply-side decision made before contacting any DSP, and it represents lost monetization. Punts are typically caused by SSP infrastructure problems, traffic quality issues, or aggressive traffic fi